# Tag 03 - Imbalanced Learning

This notebook solves the Day 3 exercises.

## Main topics

1. Imbalanced Data
2. Accuracy problem
3. Confusion Matrix
4. Precision, Recall, F1-score
5. ROC Curve and AUC
6. Threshold analysis
7. Random Oversampling
8. Random Undersampling
9. SMOTE
10. Class Weight / Cost-Sensitive Learning

This notebook is self-contained and does not require external CSV files.

In [ ]:
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import Counter

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    roc_auc_score
)

from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler

In [ ]:
# Project paths

BASE_DIR = Path.cwd()

if BASE_DIR.name == "exercise":
    BASE_DIR = BASE_DIR.parent

OUTPUT_DIR = BASE_DIR / "output"
DATA_DIR = BASE_DIR / "data"

OUTPUT_DIR.mkdir(exist_ok=True)
DATA_DIR.mkdir(exist_ok=True)

print("Base folder:", BASE_DIR)
print("Output folder:", OUTPUT_DIR)

In [ ]:
def save_plot(filename):
    path = OUTPUT_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=150)
    plt.show()
    print("Saved:", path)

def evaluate_classifier(name, y_true, y_pred, y_prob):
    return {
        "Model": name,
        "Accuracy": accuracy_score(y_true, y_pred),
        "Precision": precision_score(y_true, y_pred, zero_division=0),
        "Recall": recall_score(y_true, y_pred, zero_division=0),
        "F1": f1_score(y_true, y_pred, zero_division=0),
        "AUC": roc_auc_score(y_true, y_prob)
    }

def plot_confusion_matrix(cm, title, filename):
    plt.figure(figsize=(5, 4))
    plt.imshow(cm)
    plt.title(title)
    plt.colorbar()
    plt.xticks([0, 1], ["Class 0", "Class 1"])
    plt.yticks([0, 1], ["Class 0", "Class 1"])
    plt.xlabel("Predicted")
    plt.ylabel("Actual")

    for i in range(2):
        for j in range(2):
            plt.text(j, i, cm[i, j], ha="center", va="center")

    save_plot(filename)

# 1. Create an imbalanced dataset

We create a binary classification dataset with a strong imbalance:

- Class 0: majority class
- Class 1: minority class

This simulates real problems like fraud detection, cancer diagnosis, anomaly detection, or credit default prediction.

In [ ]:
X, y = make_classification(
    n_samples=6000,
    n_features=20,
    n_informative=8,
    n_redundant=4,
    n_clusters_per_class=2,
    weights=[0.95, 0.05],
    flip_y=0.01,
    random_state=42
)

print("Class distribution:", Counter(y))

df = pd.DataFrame(X, columns=[f"feature_{i}" for i in range(X.shape[1])])
df["target"] = y

display(df.head())

df.to_csv(DATA_DIR / "day03_synthetic_imbalanced_data.csv", index=False)

In [ ]:
class_counts = pd.Series(y).value_counts().sort_index()

plt.figure(figsize=(5, 4))
plt.bar(["Class 0 - Majority", "Class 1 - Minority"], class_counts.values)
plt.title("Original Class Distribution")
plt.ylabel("Number of samples")
save_plot("day03_original_class_distribution.png")

display(class_counts)

# 2. Train/Test Split

Important:
We use `stratify=y` so that train and test keep the same class ratio.

This is very important for imbalanced data.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

print("Train distribution:", Counter(y_train))
print("Test distribution:", Counter(y_test))

# 3. Baseline model

We train a Logistic Regression without any special handling for imbalanced data.

This shows why accuracy can be misleading.

In [ ]:
baseline_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])

baseline_model.fit(X_train, y_train)

y_pred_baseline = baseline_model.predict(X_test)
y_prob_baseline = baseline_model.predict_proba(X_test)[:, 1]

baseline_metrics = evaluate_classifier(
    "Baseline Logistic Regression",
    y_test,
    y_pred_baseline,
    y_prob_baseline
)

display(pd.DataFrame([baseline_metrics]))

print(classification_report(y_test, y_pred_baseline, zero_division=0))

In [ ]:
cm_baseline = confusion_matrix(y_test, y_pred_baseline)
print(cm_baseline)

plot_confusion_matrix(
    cm_baseline,
    "Confusion Matrix - Baseline",
    "day03_confusion_matrix_baseline.png"
)

# Important interpretation

Accuracy can look high because the majority class dominates.

Example:
If 95% of the data is class 0, a model can predict class 0 most of the time and still get high accuracy.

For imbalanced data, we need:

- Precision
- Recall
- F1-score
- ROC AUC
- Confusion Matrix

# 4. Class Weight / Cost-Sensitive Learning

Now we tell the model that minority class errors are more important.

In scikit-learn:

`class_weight="balanced"`

This does not change the data.
It changes the loss/cost function of the algorithm.

In [ ]:
weighted_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ))
])

weighted_model.fit(X_train, y_train)

y_pred_weighted = weighted_model.predict(X_test)
y_prob_weighted = weighted_model.predict_proba(X_test)[:, 1]

weighted_metrics = evaluate_classifier(
    "Weighted Logistic Regression",
    y_test,
    y_pred_weighted,
    y_prob_weighted
)

display(pd.DataFrame([weighted_metrics]))

print(classification_report(y_test, y_pred_weighted, zero_division=0))

In [ ]:
cm_weighted = confusion_matrix(y_test, y_pred_weighted)
print(cm_weighted)

plot_confusion_matrix(
    cm_weighted,
    "Confusion Matrix - Weighted Logistic Regression",
    "day03_confusion_matrix_weighted.png"
)

# 5. Random Undersampling

Random undersampling reduces the majority class.

Advantage:
- Makes the dataset balanced

Risk:
- Information loss, because many majority samples are removed

In [ ]:
rus = RandomUnderSampler(random_state=42)
X_train_under, y_train_under = rus.fit_resample(X_train, y_train)

print("Before undersampling:", Counter(y_train))
print("After undersampling:", Counter(y_train_under))

In [ ]:
under_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])

under_model.fit(X_train_under, y_train_under)

y_pred_under = under_model.predict(X_test)
y_prob_under = under_model.predict_proba(X_test)[:, 1]

under_metrics = evaluate_classifier(
    "Random Undersampling + Logistic Regression",
    y_test,
    y_pred_under,
    y_prob_under
)

display(pd.DataFrame([under_metrics]))

print(classification_report(y_test, y_pred_under, zero_division=0))

# 6. Random Oversampling

Random oversampling increases the minority class by duplicating minority examples.

Advantage:
- No majority data is removed

Risk:
- Overfitting, because duplicated samples can be memorized by the model

In [ ]:
ros = RandomOverSampler(random_state=42)
X_train_over, y_train_over = ros.fit_resample(X_train, y_train)

print("Before oversampling:", Counter(y_train))
print("After oversampling:", Counter(y_train_over))

In [ ]:
over_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])

over_model.fit(X_train_over, y_train_over)

y_pred_over = over_model.predict(X_test)
y_prob_over = over_model.predict_proba(X_test)[:, 1]

over_metrics = evaluate_classifier(
    "Random Oversampling + Logistic Regression",
    y_test,
    y_pred_over,
    y_prob_over
)

display(pd.DataFrame([over_metrics]))

print(classification_report(y_test, y_pred_over, zero_division=0))

# 7. SMOTE

SMOTE creates synthetic minority samples instead of simply duplicating existing minority samples.

It creates new points between minority class samples and their nearest neighbors.

This often works better than random oversampling.

In [ ]:
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", Counter(y_train))
print("After SMOTE:", Counter(y_train_smote))

In [ ]:
smote_model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])

smote_model.fit(X_train_smote, y_train_smote)

y_pred_smote = smote_model.predict(X_test)
y_prob_smote = smote_model.predict_proba(X_test)[:, 1]

smote_metrics = evaluate_classifier(
    "SMOTE + Logistic Regression",
    y_test,
    y_pred_smote,
    y_prob_smote
)

display(pd.DataFrame([smote_metrics]))

print(classification_report(y_test, y_pred_smote, zero_division=0))

In [ ]:
cm_smote = confusion_matrix(y_test, y_pred_smote)
print(cm_smote)

plot_confusion_matrix(
    cm_smote,
    "Confusion Matrix - SMOTE",
    "day03_confusion_matrix_smote.png"
)

# 8. Compare all methods

We compare:

1. Baseline Logistic Regression
2. Weighted Logistic Regression
3. Random Undersampling
4. Random Oversampling
5. SMOTE

For imbalanced data, focus especially on Recall, F1, and AUC.

In [ ]:
all_results = pd.DataFrame([
    baseline_metrics,
    weighted_metrics,
    under_metrics,
    over_metrics,
    smote_metrics
])

all_results = all_results.sort_values("F1", ascending=False)

display(all_results)

all_results.to_csv(OUTPUT_DIR / "day03_model_comparison.csv", index=False)

In [ ]:
metrics_to_plot = ["Accuracy", "Precision", "Recall", "F1", "AUC"]

for metric in metrics_to_plot:
    plt.figure(figsize=(10, 4))
    plt.bar(all_results["Model"], all_results[metric])
    plt.title(f"Model Comparison - {metric}")
    plt.ylabel(metric)
    plt.ylim(0, 1)
    plt.xticks(rotation=30, ha="right")
    save_plot(f"day03_model_comparison_{metric}.png")

# 9. ROC Curve and AUC

ROC Curve shows the relation between:

- TPR / Recall
- FPR

AUC summarizes the model performance across all thresholds.

In [ ]:
roc_data = {
    "Baseline": y_prob_baseline,
    "Weighted": y_prob_weighted,
    "Undersampling": y_prob_under,
    "Oversampling": y_prob_over,
    "SMOTE": y_prob_smote
}

plt.figure(figsize=(7, 6))

for name, y_prob in roc_data.items():
    fpr, tpr, thresholds = roc_curve(y_test, y_prob)
    auc_value = roc_auc_score(y_test, y_prob)
    plt.plot(fpr, tpr, label=f"{name} AUC={auc_value:.3f}")

plt.plot([0, 1], [0, 1], linestyle="--", label="Random")
plt.title("ROC Curve Comparison")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate / Recall")
plt.legend()
save_plot("day03_roc_curve_comparison.png")

# 10. Manual Threshold Analysis

A classifier often outputs probabilities.

Example:
- probability >= 0.5 => class 1
- probability < 0.5 => class 0

If we lower the threshold, recall usually increases.
But false positives usually increase too.

This is exactly the logic behind ROC curves.

In [ ]:
def threshold_metrics(y_true, y_prob, threshold):
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    tpr = tp / (tp + fn) if (tp + fn) > 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0

    return {
        "Threshold": threshold,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
        "TPR_Recall": tpr,
        "FPR": fpr,
        "Precision": precision,
        "F1": f1_score(y_true, y_pred, zero_division=0)
    }

thresholds = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]

threshold_table = pd.DataFrame([
    threshold_metrics(y_test, y_prob_smote, threshold)
    for threshold in thresholds
])

display(threshold_table)

threshold_table.to_csv(OUTPUT_DIR / "day03_threshold_analysis_smote.csv", index=False)

In [ ]:
plt.figure(figsize=(7, 5))
plt.plot(threshold_table["FPR"], threshold_table["TPR_Recall"], marker="o")

for _, row in threshold_table.iterrows():
    plt.text(row["FPR"], row["TPR_Recall"], str(row["Threshold"]))

plt.title("Manual ROC Points from Thresholds - SMOTE Model")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate / Recall")
save_plot("day03_manual_threshold_roc_points.png")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(threshold_table["Threshold"], threshold_table["TPR_Recall"], marker="o", label="Recall / TPR")
plt.plot(threshold_table["Threshold"], threshold_table["FPR"], marker="o", label="FPR")
plt.plot(threshold_table["Threshold"], threshold_table["Precision"], marker="o", label="Precision")
plt.title("Effect of Threshold on Metrics")
plt.xlabel("Threshold")
plt.ylabel("Metric value")
plt.legend()
save_plot("day03_threshold_effect_on_metrics.png")

# Day 3 Summary

Key points:

- Imbalanced data means one class appears much more often than another.
- Accuracy can be misleading for imbalanced classification.
- Confusion Matrix shows TP, FP, FN, TN.
- Recall is important when missing the positive class is dangerous.
- Precision is important when false alarms are costly.
- F1 balances Precision and Recall.
- ROC curve shows TPR vs FPR across thresholds.
- AUC summarizes overall model performance.
- Random undersampling can lose information.
- Random oversampling can cause overfitting.
- SMOTE creates synthetic minority examples.
- Class weighting changes the loss function without changing the data.